# Instamart Customer Intelligence & Support Copilot
## End-to-End Capstone Analysis Notebook

**Project:** IIT Patna Applied AI/ML Capstone  
**Scenario:** Synthetic grocery quick-commerce data for Instamart.

This notebook is a supplementary inspection notebook for the Python pipeline. It reads the latest files generated by `part_a_pipeline.py`, `part_b_pipeline.py`, `part_c_pipeline.py`, and `part_d_pipeline.py`. The Python scripts remain the source of truth for the project implementation.

**Important:** Run `python run_project.py` from the project folder first. Then run every notebook cell from top to bottom to refresh the notebook's displayed tables and outputs.


In [6]:
from pathlib import Path
import sys
import json
import sqlite3
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "generate_data.py").exists():
    candidates = [Path.cwd().parent, Path.cwd().parent.parent]
    ROOT = next((p for p in candidates if (p / "generate_data.py").exists()), ROOT)

if not (ROOT / "generate_data.py").exists():
    raise FileNotFoundError(
        "Project root not found. Open this notebook from the instamart-copilot folder "
        "or update ROOT to the folder containing generate_data.py."
    )

OUTPUTS = ROOT / "outputs"
DATA = ROOT / "data"
sys.path.insert(0, str(ROOT))

print("Project root:", ROOT)
print("Notebook:", Path("Instamart_Copilot_Analysis.ipynb").name)
print("Pipeline scripts:", all((ROOT / f"part_{p}_pipeline.py").exists() for p in "abcd"))
print("Outputs folder exists:", OUTPUTS.exists())


Project root: c:\Users\Admin\Desktop\instamart-copilot
Notebook: Instamart_Copilot_Analysis.ipynb
Pipeline scripts: True
Outputs folder exists: True


## Part A — Data Foundation, Cleaning & SQLite Storage

The Part A pipeline standardizes customer fields, removes duplicate orders, resolves order-level records, cleans numeric values, handles invalid delivery times, and stores the cleaned tables in SQLite.

The latest verified pipeline run produced:
- 1,200 customers, 40 products, and 12,309 cleaned orders.
- 11,110 completed orders and 1,199 cancelled orders; no pending orders remained.
- Completed-order revenue of **₹29,98,668.80**.
- Completed orders with missing delivery time after cleaning: **0**. Cancelled orders retain missing delivery times.
- Part A validation: **31 checks passed, 0 failed**.


In [7]:
db_path = DATA / "instamart.db"
if not db_path.exists():
    raise FileNotFoundError("SQLite database not found. Run python run_project.py first.")

with sqlite3.connect(db_path) as con:
    tables = pd.read_sql_query(
        "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", con
    )
    table_counts = pd.read_sql_query(
        "SELECT 'customers' AS table_name, COUNT(*) AS row_count FROM customers "
        "UNION ALL SELECT 'products', COUNT(*) FROM products "
        "UNION ALL SELECT 'orders', COUNT(*) FROM orders",
        con
    )
    order_summary = pd.read_sql_query(
        "SELECT status, COUNT(*) AS order_count, "
        "ROUND(SUM(CASE WHEN status='completed' THEN order_amount ELSE 0 END), 2) "
        "AS completed_revenue_inr "
        "FROM orders GROUP BY status ORDER BY status",
        con
    )
    revenue_by_city = pd.read_sql_query(
        "SELECT c.city, ROUND(SUM(o.order_amount), 2) AS revenue_inr "
        "FROM orders o JOIN customers c USING(customer_id) "
        "WHERE o.status='completed' GROUP BY c.city ORDER BY revenue_inr DESC",
        con
    )

display(tables)
display(table_counts)
display(order_summary)
display(revenue_by_city)
print("Latest verified completed-order revenue (INR): ₹29,98,668.80")


,name
0,customers
1,orders
2,products


,table_name,row_count
0,customers,1200
1,products,40
2,orders,12309


,status,order_count,completed_revenue_inr
0,cancelled,1199,0.0
1,completed,11110,2998668.8


,city,revenue_inr
0,Hyderabad,528589.90
1,Bengaluru,523643.40
2,Delhi,502829.80
3,Pune,500918.05
4,Mumbai,479058.50
5,Chennai,463629.15


Latest verified completed-order revenue (INR): ₹29,98,668.80


## Part B — Customer Segmentation & Lapse Prediction

The latest verified segmentation uses four K-means clusters, with the project labels:
**Premium**, **Frequent**, **At Risk**, and **At Risk 3**.

The selected lapse-prediction model is **Logistic Regression**, with test F1 **0.7236**. Random Forest was evaluated and its feature importances were inspected; it was not the selected final model.

Latest verified evaluation:
- Logistic Regression test precision: **0.6545**
- Logistic Regression test recall: **0.8090**
- Logistic Regression test F1: **0.7236**
- Random Forest test F1: **0.7059**
- PCA variance explained by PC1 and PC2 together: **0.8523**


In [8]:
segments_path = OUTPUTS / "customer_segments.csv"
profiles_path = OUTPUTS / "segment_profiles.csv"
retention_path = OUTPUTS / "retention_list.csv"
metrics_path = OUTPUTS / "lapse_model_metrics.csv"
kmeans_path = OUTPUTS / "kmeans_evaluation.csv"

for path in [segments_path, profiles_path, retention_path, metrics_path, kmeans_path]:
    if not path.exists():
        print("Missing output:", path.name, "- run python run_project.py first.")

if segments_path.exists():
    segments_df = pd.read_csv(segments_path)
    print("Customer segment sample:")
    display(segments_df.head())
    if "segment" in segments_df.columns:
        print("Segment counts:")
        display(segments_df["segment"].value_counts().rename_axis("segment").reset_index(name="customers"))

if profiles_path.exists():
    print("Segment profiles:")
    display(pd.read_csv(profiles_path))

if metrics_path.exists():
    print("Lapse model metrics:")
    display(pd.read_csv(metrics_path))

if kmeans_path.exists():
    print("K-means evaluation:")
    display(pd.read_csv(kmeans_path))

if retention_path.exists():
    print("Retention list sample:")
    display(pd.read_csv(retention_path).head(10))


Customer segment sample:


,customer_id,recency,frequency,monetary,segment_id,segment_name,is_premium
0,C0001,14,4,512.85,0,At Risk,False
1,C0002,28,7,4644.90,1,Premium,True
2,C0003,7,22,3644.85,2,Frequent,False
3,C0004,0,5,369.00,0,At Risk,False
4,C0005,147,2,324.00,3,At Risk 3,False


Segment profiles:


,segment_id,customer_count,mean_recency,mean_frequency,mean_monetary,segment_name,is_premium
0,1,138,24.115942,6.768116,7129.695652,Premium,True
1,2,419,8.766110,19.403341,3410.020525,Frequent,False
2,0,436,28.862385,3.772936,1057.618234,At Risk,False
3,3,182,109.043956,2.203297,685.992582,At Risk 3,False


Lapse model metrics:


,model,precision,recall,f1
0,Baseline,0.000000,0.000000,0.000000
1,Logistic Regression,0.654545,0.808989,0.723618
2,Random Forest,0.673469,0.741573,0.705882


K-means evaluation:


,k,inertia,silhouette_score
0,2,1820.760553,0.464972
1,3,1268.234587,0.445161
2,4,743.071108,0.525544
3,5,617.366636,0.480495
4,6,503.170622,0.481872
5,7,408.460454,0.416337
6,8,366.503487,0.412329


Retention list sample:


,customer_id,lapse_probability,actually_lapsed
0,C0577,0.574938,0
1,C0102,0.570068,1
2,C0697,0.547085,0
3,C0942,0.509633,0
4,C0831,0.495896,0
5,C0324,0.476695,0
6,C0162,0.476268,0
7,C0509,0.476204,0
8,C0977,0.445404,1
9,C0299,0.445151,0


## Part C — Support Ticket Triage & Privacy Guardrails

The Part C pipeline masks personally identifiable information before sending ticket text to the supplied offline mock LLM. It evaluates zero-shot and few-shot classification, applies input/output guardrails, tracks token usage, and routes high-risk fraud cases for human review.

Latest verified results:
- Total tickets: **164**
- PII patterns masked: **278** (55 email, 91 phone, 132 order-ID matches)
- Zero-shot accuracy: **78.75%**
- Few-shot accuracy: **91.875%**
- Few-shot improvement: **13.125 percentage points**
- Human-review queue: **17 tickets**
- Fraud tickets assigned to Security Team: **34**, all marked HIGH priority.


In [9]:
triage_path = OUTPUTS / "triage_results.csv"
zero_path = OUTPUTS / "triage_zero_shot_results.csv"
few_path = OUTPUTS / "triage_fewshot_comparison.csv"
review_path = OUTPUTS / "triage_human_review.csv"
fraud_path = OUTPUTS / "triage_fraud_security.csv"
budget_path = OUTPUTS / "triage_budget_demo.csv"

for path in [triage_path, zero_path, few_path, review_path, fraud_path, budget_path]:
    if not path.exists():
        print("Missing output:", path.name, "- run python run_project.py first.")

if triage_path.exists():
    triage_df = pd.read_csv(triage_path)
    print("Final triage records:", len(triage_df))
    display(triage_df.head(10))
    if "predicted_category" in triage_df.columns:
        print("Predicted category counts:")
        display(triage_df["predicted_category"].value_counts().rename_axis("category").reset_index(name="tickets"))

if zero_path.exists():
    print("Zero-shot result sample:")
    display(pd.read_csv(zero_path).head())

if few_path.exists():
    print("Zero-shot versus few-shot comparison:")
    display(pd.read_csv(few_path))

if review_path.exists():
    print("Human-review queue:")
    display(pd.read_csv(review_path).head(10))

if fraud_path.exists():
    print("Fraud cases routed to Security Team:")
    display(pd.read_csv(fraud_path).head(10))

if budget_path.exists():
    print("Token-budget demonstration:")
    display(pd.read_csv(budget_path))


Final triage records: 164


,ticket_id,customer_id,true_category,predicted_category,reason,status,priority,assigned_team,email_redactions,phone_redactions,order_id_redactions,llm_calls,prompt_tokens,completion_tokens,total_tokens
0,T0024,C0486,fraud,fraud,keyword match,classified_json,HIGH,Security Team,0,1,1,1,80,7,87
1,T0073,C0080,billing,billing,keyword match,classified_json,NORMAL,Customer Support,0,1,1,1,76,7,83
2,T0095,C1087,technical,technical,keyword match,classified_json,NORMAL,Customer Support,0,0,1,1,78,7,85
3,T0159,C1172,none,human_review,empty ticket sent for human review,skipped_empty,NORMAL,Customer Support,0,0,0,0,0,0,0
4,T0080,C0910,fraud,fraud,keyword match,classified_json,HIGH,Security Team,0,1,1,1,80,7,87
5,T0124,C0139,fraud,fraud,keyword match,classified_json,HIGH,Security Team,0,1,1,1,80,7,87
6,T0061,C0812,billing,billing,keyword match,classified_json,NORMAL,Customer Support,1,0,1,1,72,7,79
7,T0118,C0355,delivery,delivery,keyword match,classified_json,NORMAL,Customer Support,1,0,1,1,75,7,82
8,T0037,C1156,billing,billing,keyword match,classified_json,NORMAL,Customer Support,1,0,1,1,76,7,83
9,T0002,C0881,delivery,delivery,keyword match,classified_json,NORMAL,Customer Support,0,1,1,2,74,7,81


Predicted category counts:


,category,tickets
0,technical,39
1,billing,38
2,delivery,36
3,fraud,34
4,human_review,17


Zero-shot result sample:


,ticket_id,customer_id,true_category,predicted_category,reason,status,priority,assigned_team,email_redactions,phone_redactions,order_id_redactions,llm_calls,prompt_tokens,completion_tokens,total_tokens
0,T0024,C0486,fraud,fraud,keyword match,classified_json,HIGH,Security Team,0,1,1,1,51,7,58
1,T0073,C0080,billing,billing,keyword match,classified_json,NORMAL,Customer Support,0,1,1,1,47,7,54
2,T0095,C1087,technical,billing,keyword match,classified_json,NORMAL,Customer Support,0,0,1,1,49,7,56
3,T0159,C1172,none,human_review,empty ticket sent for human review,skipped_empty,NORMAL,Customer Support,0,0,0,0,0,0,0
4,T0080,C0910,fraud,billing,keyword match,classified_json,NORMAL,Customer Support,0,1,1,1,51,7,58


... (4 outputs left out)

Fraud cases routed to Security Team:


,ticket_id,customer_id,true_category,predicted_category,reason,status,priority,assigned_team,email_redactions,phone_redactions,order_id_redactions,llm_calls,prompt_tokens,completion_tokens,total_tokens
0,T0024,C0486,fraud,fraud,keyword match,classified_json,HIGH,Security Team,0,1,1,1,80,7,87
1,T0080,C0910,fraud,fraud,keyword match,classified_json,HIGH,Security Team,0,1,1,1,80,7,87
2,T0124,C0139,fraud,fraud,keyword match,classified_json,HIGH,Security Team,0,1,1,1,80,7,87
3,T0136,C0891,fraud,fraud,keyword match,classified_json,HIGH,Security Team,1,0,0,1,74,7,81
4,T0008,C1177,fraud,fraud,keyword match,classified_json,HIGH,Security Team,0,1,1,1,80,7,87
5,T0072,C0218,fraud,fraud,keyword match,classified_json,HIGH,Security Team,0,1,1,1,80,7,87
6,T0156,C0328,fraud,fraud,keyword match,classified_json,HIGH,Security Team,0,1,1,1,80,7,87
7,T0152,C0530,fraud,fraud,keyword match,classified_json,HIGH,Security Team,0,1,1,2,80,7,87
8,T0144,C0521,fraud,fraud,keyword match,classified_json,HIGH,Security Team,0,1,1,1,80,7,87
9,T0036,C0173,fraud,fraud,keyword match,classified_json,HIGH,Security Team,0,1,1,1,80,7,87


Token-budget demonstration:


,ticket_id,customer_id,true_category,predicted_category,reason,status,priority,assigned_team,email_redactions,phone_redactions,order_id_redactions,llm_calls,prompt_tokens,completion_tokens,total_tokens
0,T0024,C0486,fraud,fraud,keyword match,classified_json,HIGH,Security Team,0,1,1,1,80,7,87
1,T0073,C0080,billing,billing,keyword match,classified_json,NORMAL,Customer Support,0,1,1,1,76,7,83
2,T0095,C1087,technical,technical,keyword match,classified_json,NORMAL,Customer Support,0,0,1,1,78,7,85
3,T0159,C1172,none,human_review,empty ticket sent for human review,skipped_empty,NORMAL,Customer Support,0,0,0,0,0,0,0
4,T0080,C0910,fraud,human_review,token budget insufficient; human review required,budget_exhausted,NORMAL,Customer Support,0,1,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
159,T0150,C0622,delivery,human_review,token budget insufficient; human review required,budget_exhausted,NORMAL,Customer Support,0,1,1,0,0,0,0
160,T0065,C0205,billing,human_review,token budget insufficient; human review required,budget_exhausted,NORMAL,Customer Support,0,1,1,0,0,0,0
161,T0128,C0467,fraud,human_review,token budget insufficient; human review required,budget_exhausted,NORMAL,Customer Support,1,0,0,0,0,0,0
162,T0005,C0404,billing,human_review,token budget insufficient; human review required,budget_exhausted,NORMAL,Customer Support,1,0,1,0,0,0,0


## Part D — Policy RAG, Tool-Using Agent & Refund Ledger

The Part D pipeline indexes five local policy documents using word-based chunks (40 words, 10-word overlap), retrieves policy context, and runs a tool-using support agent.

The agent tools are `get_order`, `search_policy`, `calculate_refund`, `issue_refund`, and `escalate_to_human`.

Latest verified results:
- 15 policy chunks were created.
- 14 policy questions were evaluated: 12 answerable and 2 unanswerable.
- All 12 answerable questions matched the expected policy facts; both unanswerable questions were correctly answered with an abstention.
- Seven agent scenarios passed.
- Refund ledger: S1 **₹128.25**, S2 **₹135.00**, total **₹263.25**.
- Checkpoint/resume for S1 completed without duplicate tool calls.

For the two unanswerable cases, the evaluation's `retrieval_match` and `fact_match` indicators represent successful abstention handling; they should not be interpreted as proof that the requested facts appeared in the policy documents.


In [10]:
policy_results_path = OUTPUTS / "policy_rag_results.csv"
policy_compare_path = OUTPUTS / "policy_rag_comparison.csv"
scenario_path = OUTPUTS / "agent_scenario_results.csv"
ledger_path = OUTPUTS / "refund_ledger.jsonl"
agent_runs_path = OUTPUTS / "agent_runs.jsonl"

for path in [policy_results_path, policy_compare_path, scenario_path, ledger_path, agent_runs_path]:
    if not path.exists():
        print("Missing output:", path.name, "- run python run_project.py first.")

if policy_results_path.exists():
    print("Policy RAG evaluation:")
    display(pd.read_csv(policy_results_path))

if policy_compare_path.exists():
    print("Policy RAG comparison:")
    display(pd.read_csv(policy_compare_path))

if scenario_path.exists():
    print("Agent scenario results:")
    display(pd.read_csv(scenario_path))

if ledger_path.exists():
    ledger_entries = []
    with open(ledger_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                ledger_entries.append(json.loads(line))
    ledger_df = pd.DataFrame(ledger_entries)
    print("Refund ledger:")
    display(ledger_df)
    if "refund_inr" in ledger_df.columns:
        print("Total refunds (INR):", round(ledger_df["refund_inr"].sum(), 2))

if agent_runs_path.exists():
    print("Agent trace sample:")
    with open(agent_runs_path, "r", encoding="utf-8") as f:
        for _, line in zip(range(3), f):
            print(line.rstrip())


Policy RAG evaluation:


,question,expected_doc,expected_fact,retrieved_documents,retrieval_match,answer,fact_match,fallback_used,chunk_configuration
0,What delivery time makes a completed order eli...,refund_policy.txt,more than 45 minutes,refund_policy.txt | refund_policy.txt | paymen...,True,"order is delivered in more than 45 minutes, th...",True,False,40_words_10_overlap
1,Who must approve a refund above INR 1000?,refund_policy.txt,human support lead,refund_policy.txt | refund_policy.txt | refund...,True,A refund above INR 1000 must be approved by a ...,True,False,40_words_10_overlap
2,How many days do I have to raise a refund requ...,refund_policy.txt,within 7 days,refund_policy.txt | privacy_policy.txt | refun...,True,A refund request must be raised within 7 days ...,True,False,40_words_10_overlap
3,What is the delivery fee on small orders?,delivery_policy.txt,INR 25,delivery_policy.txt | delivery_policy.txt | pa...,True,A delivery fee of INR 25 is charged on orders ...,True,False,40_words_10_overlap
4,On orders of what value is delivery free?,delivery_policy.txt,INR 199 or more,delivery_policy.txt | delivery_policy.txt | pa...,True,"of INR 25 is charged on orders below INR 199, ...",True,False,40_words_10_overlap
5,Why does the electronics delivery need a one t...,delivery_policy.txt,one time password,delivery_policy.txt | delivery_policy.txt | pa...,True,Electronics items are delivered only after the...,True,False,40_words_10_overlap
6,Is cash on delivery available for a large order?,payment_policy.txt,INR 2500,payment_policy.txt | payment_policy.txt | paym...,True,Cash on delivery is available only for orders ...,True,False,40_words_10_overlap
7,How soon is money returned when a payment fail...,payment_policy.txt,48 hours,payment_policy.txt | payment_policy.txt | refu...,True,"If a payment fails but money is deducted, the ...",True,False,40_words_10_overlap
8,Can wallet balance be refunded as cash?,payment_policy.txt,never refunded as cash,payment_policy.txt | payment_policy.txt | paym...,True,Wallet balance can be used for any order and i...,True,False,40_words_10_overlap
9,When are delivery addresses deleted from activ...,privacy_policy.txt,180 days,privacy_policy.txt | privacy_policy.txt | deli...,True,Delivery addresses are deleted from active sys...,True,False,40_words_10_overlap


Policy RAG comparison:


,configuration,chunks,retrieval_matches,fact_matches
0,"40 words, 10 overlap",15,14,14
1,"40 words, 0 overlap",13,14,12


Agent scenario results:


,scenario_id,expected_outcome,expected_refund_inr,actual_response,passed
0,S1,REFUND,128.25,REFUND: INR 128.25 issued.,True
1,S2,REFUND,135.00,REFUND: INR 135.0 issued.,True
2,S3,ESCALATE,1999.00,ESCALATE: refund above INR 1000 sent to a huma...,True
3,S4,DENIED,0.00,DENIED: delivery was within 45 minutes.,True
4,S5,DENIED,0.00,DENIED: only completed orders can be refunded.,True
5,S6,NOT_FOUND,0.00,NOT_FOUND: no such order.,True
6,S7,FALLBACK,0.00,FALLBACK: Maximum iteration limit reached. Han...,True


Refund ledger:


,scenario_id,customer_id,refund_inr
0,S1,C0007,128.25
1,S2,C0008,135.00


Total refunds (INR): 263.25
Agent trace sample:
{"scenario_id": "S1", "steps": [{"step": 1, "tool_call": {"name": "get_order", "arguments": {"order_id": "ORD-100048"}}}, {"step": 2, "tool_call": {"name": "search_policy", "arguments": {"query": "refund policy late delivery"}}}, {"step": 3, "tool_call": {"name": "calculate_refund", "arguments": {"order_amount": 128.25, "delivery_minutes": 65.0}}}, {"step": 4, "tool_call": {"name": "issue_refund", "arguments": {"order_id": "ORD-100048", "refund_inr": 128.25}}}, {"step": 5, "final_response": "REFUND: INR 128.25 issued."}], "final_response": "REFUND: INR 128.25 issued.", "tool_call_counts": {"get_order": 1, "search_policy": 1, "calculate_refund": 1, "issue_refund": 1}, "tool_observations": 4}
{"scenario_id": "S2", "steps": [{"step": 1, "tool_call": {"name": "get_order", "arguments": {"order_id": "ORD-100067"}}}, {"step": 2, "tool_call": {"name": "search_policy", "arguments": {"query": "refund policy late delivery"}}}, {"step": 3, "tool_call

## Limitations and Next Steps

- The project uses synthetic data and a deterministic, offline mock LLM; results do not establish production performance.
- The lapse model and ticket classifier are evaluated on this generated dataset. Their metrics should not be generalized to real customers or support traffic.
- Policy retrieval depends on the coverage and wording of the supplied policy documents; the assistant appropriately abstains when required information is unavailable.
- The agent's retry, timeout, token-budget, and iteration-limit demonstrations are controlled test scenarios, not measured production service-level limits.
- A production deployment would require representative real-world validation, monitoring, access controls, audit logging, and human review for sensitive or high-impact decisions.

**Reproduction:** From the project root, activate the environment, install `requirements.txt`, run `python run_project.py`, then execute all cells in this notebook and save it.
